<a href="https://colab.research.google.com/github/Btnunes/Fligthtops_Analytics_br/blob/main/Flightops_Analytics_BR_v1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gtts --quiet

In [ ]:
import time
import requests
import pandas as pd
import folium
from IPython.display import display
from IPython.display import Audio

from gtts import gTTS

from datetime import datetime
from zoneinfo import ZoneInfo

agora = datetime.now(ZoneInfo('America/Sao_Paulo'))

# ==============================================================================
# FUNÇÃO METEOROLÓGICA - OPEN METEO
# ==============================================================================

def obter_clima(lat, long):
  try:
    url_clima = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={long}"
        f"&current_weather=true"
    )
    response = requests.get(url_clima, timeout=20)

    dados_clima = response.json()

    clima = dados_clima.get('current_weather', {})
    temperatura = clima.get('temperature', 'N/A')
    vento = clima.get('windspeed', 'N/A')
    codigo = clima.get('weathercode', -1)

    print("Tipo do weathercode:", type(codigo))
    print("Valor do wethercode:", codigo)

    return {
        "temperatura": temperatura,
        "vento": vento,
        "codigo": codigo
    }
  except Exception as erro:
    print(f"Erro ao obter dados meteorológicos:", erro)

  return {
      "temperatura": "N/A",
      "vento": "N/A",
      "codigo": -1
  }

# ==============================================================================
# DECODIFICAÇÃO METEOROLÓGICA
# ==============================================================================

def traduzir_weathercode(codigo):

  weathercodes = {

      0: "Céu limpo",
      1: "Principalmente limpo",
      2: "Parcialmente nublado",
      3: "Nublado",
      45: "Neblina",
      48: "Neblina congelante",
      51: "Garoa leve",
      53: "Garoa moderada",
      55: "Garoa intensa",
      61: "Chuva leve",
      63: "Chuva moderada",
      65: "Chuva forte",
      71: "Neve leve",
      80: "Pancadas leves",
      81: "Pancadas moderadas",
      82: "Pancadas fortes",
      95: "Tempestade"

  }
  return weathercodes.get(codigo, "Condição desconhecida")

#===============================================================================
# CONVERSÃO DE GALLSIGN PARA FONIA AERONÁUTICA BRASILEIRA
#===============================================================================
def fonia_callsign(callsign):
  callsign = str(callsign).strip()

  empresas = {
      "AZU": "Azul",
      "GLO": "Gol",
      "TAM": "Latam"
  }

  numeros = {
      "0": "zero",
      "1": "uno",
      "2": "dois",
      "3": "três",
      "4": "quatro",
      "5": "cinco",
      "6": "meia",
      "7": "sete",
      "8": "oito",
      "9": "nove"
  }

  prefixo = callsign[:3]
  numeros_voo = callsign[3:]

  nome_empresa = empresas.get(
      prefixo,
      prefixo
  )

  numeros_convertidos = []

  for numero in numeros_voo:
    numeros_convertidos.append(
        numeros.get(numero, numero)
    )

  return(
      nome_empresa + " " + " ".join(numeros_convertidos)
  )

# ==========================================================
# CONVERSÃO IATA → CIDADE PARA VOZ SINTETIZADA
# ==========================================================

cidades_fonia = {

    "BSB": "Brasília",
    "CGH": "São Paulo, Congonhas",
    "GRU": "São Paulo, Guarulhos",
    "VCP": "Campinas",
    "REC": "Recife",
    "SSA": "Salvador",
    "SDU": "Rio de Janeiro, Santos Dumont",
    "GIG": "Rio de Janeiro, Galeão",
    "CNF": "Confins",
    "POA": "Porto Alegre",
    "FOR": "Fortaleza",
    "BEL": "Belém",
    "NAT": "Natal",
    "FLN": "Florianópolis",
    "GYN": "Goiânia",
    "CGB": "Cuiabá",
    "SLZ": "São Luís",
    "MCZ": "Maceió",
    "JPA": "João Pessoa",
    "IOS": "Ilhéus",
    "VIX": "Vitória",
    "PMW": "Palmas",
    "RAO": "Ribeirão Preto",
    "LDB": "Londrina",
    "NVT": "Navegantes",
    "IGU": "Foz do Iguaçu",
    "AEP": "Buenos Aires",
    "MIA": "Miami",
    "LIS": "Lisboa",
    "OPS": "Sinop"

}


#===============================================================================
# FUNÇÃO DE VOZ SINTETIZADA
#===============================================================================
def falar_fonia(texto, arquivo):
  tts = gTTS(
      text=texto,
      lang='pt',

  )
  tts.save(arquivo)

  return Audio(arquivo)

#===============================================================================
# FUNÇÃO DE FONIA OPERACIONAL
#===============================================================================
def gerar_fonia(callsign, altitude, velocidade, rota):

    try:

        callsign = str(callsign).strip()

        callsign_fonia = fonia_callsign(
            callsign
        )

        if "→" in rota:

            origem, destino = rota.split("→")

            destino = destino.strip()

            destino_fonia = cidades_fonia.get(
                destino,
                destino
            )

        else:

            destino = "destino desconhecido"

            destino_fonia = destino

        orgao_app = orgaos_ats.get(

            destino,

            "Controle Aproximação"
        )
    #===========================================================================
    # FASE DO VOO: ALTITUDE DE CRUZEIRO
    #===========================================================================

        if altitude >= 30000:

            return(
                f"🎧 ACC:<br>"
                f"{callsign_fonia}, "
                f"mantenha nível "
                f"{int(altitude/100)}, "
                f"prossiga {destino_fonia}."
            )

    #===========================================================================
    # FASE DO VOO: SUBIDA
    #===========================================================================

        elif altitude >= 10000 and velocidade >= 650:

            return(
                f"🎧 PILOTO:<br>"
                f"Centro Brasília,<br>"
                f"{callsign_fonia}, "
                f"passando nível "
                f"{int(altitude/100)}, "
                f"subindo para nível "
                f"{int((altitude + 5000)/100)}.<br><br>"

                f"🎧 ACC:<br>"
                f"{callsign_fonia}, "
                f"Centro Brasília, "
                f"contato radar, "
                f"prossiga conforme autorizado."
            )

    #===========================================================================
    # FASE DO VOO: APROXIMAÇÃO
    #===========================================================================

        elif altitude >= 3000:

            return(
                f"🎧 PILOTO:<br>"
                f"{orgao_app},<br>"
                f"{callsign_fonia}, "
                f"nível {int(altitude/100)}, "
                f"aproximando-se de {destino_fonia}, "
                f"informação Charlie.<br><br>"

                f"🎧 APP:<br>"
                f"{callsign_fonia}, "
                f"aproximação radar contato, "
                f"prossiga para pouso,<br><br>"

                f"🎧 TORRE:<br>"
                f"{callsign_fonia}, "
                f"vento 140 graus, "
                f"mantenha alinhamento, "
                f"autorizado pouso pista 15."


            )

    #===========================================================================
    # FASE DO VOO: FINAL / POUSO
    #===========================================================================
        else:

            return(
                f"🎧 TORRE:<br>"
                f"{callsign_fonia}, "
                f"autorizado pouso."
                f"para {destino_fonia}."
            )

    except Exception as erro:
      return f"Fonia indisponível: {erro}"

# ==============================================================================
# FUNÇÃO PARA DESENHAR ROTA DINÂMICA
# ==============================================================================
def desenhar_rota(
    mapa,
    rota,
    latitude_atual,
    longitude_atual
):

    try:

        origem, destino = rota.split(" → ")

        if origem in coordenadas_rotas:

            ponto_origem = coordenadas_rotas[origem]

            ponto_atual = [
                latitude_atual,
                longitude_atual
            ]

            folium.PolyLine(
                locations=[
                    ponto_origem,
                    ponto_atual
                ],
                color='blue',
                weight=2,
                opacity=0.7
            ).add_to(mapa)

    except Exception as erro:
        print(f"Erro ao desenhar rota: {erro}")

# ==============================================================================
# CONFIGURAÇÃO VISUAL
# ==============================================================================

RED = "\033[1;31m"
GREEN = "\033[1;32m"
YELLOW = "\033[1;33m"
BLUE = "\033[1;34m"
CYAN = "\033[1;36m"
WHITE = "\033[1;37m"
RESET = "\033[0m"
BOLD = "\033[1m"

# ==============================================================================
# CABEÇALHO DO SISTEMA
# ==============================================================================

print(f"{CYAN}{'='*92}{RESET}")
print(f"{BOLD}FLIGHTOPS ANALYTICS BR v1.2 ✈{RESET}")
print(f"{CYAN}EDIÇÃO CLIMA OPERACIONAL{RESET}")
print(f"{CYAN}CENTRO OPERACIONAL AÉREO EM TEMPO REAL{RESET}")
print(f"{CYAN}Sistema de Monitoramento Aeronáutico com Inteligência Meteorológica{RESET}")
print(f"{CYAN}{'-'*92}{RESET}")
print(f"{BOLD}Desenvolvido por Carlos Alberto Nunes{RESET}")
print(f"{CYAN}Local de execução: Google Colab{RESET}")
print(f"{CYAN}Timezone: America / São Paulo{RESET}")
print(f"{CYAN}{'-'*92}{RESET}")
print(f"Consulta realizada em: {agora.strftime('%d/%m/%Y %H:%M:%S')}")
print(f"{CYAN}{'-'*92}{RESET}")
print(f"{BOLD}Fontes de dados integradas:{RESET}")
print(f"{CYAN}◊ OpenSky Network API → tráfego aéreo em tempo real{RESET}")
print(f"{CYAN}◊ Open Meteo API → dados meteorológicos em tempo real{RESET}")
print(f"{CYAN}{'-'*92}{RESET}")
print(f"{CYAN}◊ Rastreamento ADS-B{RESET}")
print(f"{CYAN}◊ Geolocalização aeronáutica{RESET}")
print(f"{CYAN}◊ Aeroportos nacionais{RESET}")
print(f"{CYAN}◊ Meteorologia operacional{RESET}")
print(f"{CYAN}{'='*92}{RESET}")

# ==============================================================================
# DATA / HORA DA CONSULTA
# ==============================================================================

agora = datetime.now()
print(f"\n🕐 Consulta realizada em: {agora.strftime('%d/%m/%Y %H:%M:%S')}")

# ==============================================================================
# CONSUMO DA API OPENSKY
# ==============================================================================

print(f"\n{YELLOW}🔄 Conectando à OpenSky NetWork API...{RESET}")

url = "https://opensky-network.org/api/states/all?lamin=-34&lomin=-55&lamax=6&lomax=-35"

try:
  headers={
      "User-Agent": "FlightOpsAnalytcsBR/1.2"
  }

  response = requests.get(
      url,
      headers=headers,
      timeout=60
  )

  print(f"STATUS HTTP: {response.status_code}")
  if response.status_code == 200:
    dados = response.json()
    if dados is not None and 'states' in dados:
      print(f"{GREEN}✅ Conexão bem-sucedida à OpenSky NetWork API.{RESET}")

    else:
      print(f"{RED}❌ API retornou dados inválidos: {RESET}")

      raise Exception("JSON inválido")

  else:
    print(f"{RED}❌ Erro HTTP: {response.status_code}{RESET}")
    raise Exception("Falha HTTP")

except Exception as erro:
  print(f"{RED}❌ Falha ao obter dados da OpenSky NetWork API: {RESET}")
  print(f"{YELLOW}Detalhes: {RESET}", erro)
  raise SystemExit("Sistema Encerrado por falha na API.")

# ==============================================================================
# ESTRUTURA DAS COLUNAS
# ==============================================================================
colunas = [
    "icao24",
    "callsign",
    "origin_country",
    "time_position",
    "last_contact",
    "longitude",
    "latitude",
    "baro_altitude",
    "on_ground",
    "velocity",
    "true_track",
    "vertical_rate",
    "sensors",
    "geo_altitude",
    "squawk",
    "spi",
    "position_source"
]

# ==============================================================================
# CRIAÇÃO DO DATAFRAME
# ==============================================================================
print(f"\n{YELLOW}🔄 Processando dados operacionais...{RESET}")

print(type(dados))
df = pd.DataFrame(dados['states'], columns = colunas)

# ==============================================================================
# LIMPEZA DOS DADOS
# ==============================================================================

df = df[
    (df['latitude'].notnull()) &
    (df['longitude'].notnull())
]

# ==============================================================================
# FILTRO DE AERONAVES NO BRASIL
# ==============================================================================

df_brasil = df[
    (df['latitude'] >= -35) &
    (df['latitude'] <= 5) &
    (df['longitude'] >= -75) &
    (df['longitude'] <= -30)
]

# ==============================================================================
# AJUSTE DE DADOS
# ==============================================================================

df_brasil = df_brasil.copy()
df_brasil['callsign'] = df_brasil['callsign'].fillna("N/A")

df_brasil['velocity_kmh'] = (
    df_brasil['velocity'] * 3.6
).round(1)

df_brasil['altitude_ft'] = (
    df_brasil['baro_altitude'] * 3.28084
).round(0)

# ==============================================================================
# ESTATÍSTICAS OPERACIONAIS
# ==============================================================================


total_aeronaves = len(df_brasil)
media_velocidade = df_brasil['velocity_kmh'].mean()
media_altitude = df_brasil['altitude_ft'].mean()

# ==============================================================================
# RESUMO OPERACIONAL
# ==============================================================================

print(f"\n{CYAN}{'='*90}{RESET}")
print(f"{BOLD} RESUMO OPERACIONAL AÉREO - BRASIL{RESET}")
print(f"\n{CYAN}{'='*90}{RESET}")

print(f"\n✈️ Aeronaves monitoradas: {GREEN}{total_aeronaves}{RESET}")

print(f"✈️ Velocidade média: {GREEN}{media_velocidade:.1f} km/h{RESET}")

print(f"✈️ Altitude média: {GREEN}{media_altitude:.0f} pés{RESET}")

# ==============================================================================
# EXIBIÇÃO DA TABELA OPERACIONAL
# ==============================================================================

print(f"\n{CYAN}{'='*90}{RESET}")
print(f"{BOLD} PAINEL OPERACIONAL DE AERONAVES{RESET}")
print(f"{CYAN}{'='*90}{RESET}")

painel = df_brasil[
  [
    'callsign',
    'origin_country',
    'velocity_kmh',
    'altitude_ft',
    'latitude',
    'longitude',
    'on_ground'

  ]
].head(30)

painel.columns = [
    'CALLSIGN',
    'PAÍS DE ORIGEM',
    'VELOCIDADE_KMH',
    'ALTITUDE_FT',
    'LATITUDE',
    'LONGITUDE',
    'EM_SOLO'
]

display(painel)

# ==============================================================================
# CRIAÇÃO DO MAPA AERONÁUTICO
# ==============================================================================

print(f"\n{YELLOW}🗺 Gerando mapa aeronáutico operacional...{RESET}")

mapa = folium.Map(
    location=[-15.78, -47.93],
    zoom_start=5,
    tiles='CartoDB positron'
)

# ==============================================================================
# ROTAS CONHECIDAS (simulação operacional)
# ==============================================================================
rotas_conhecidas = {
    "AZU5093": "VIX → VCP",
    "AZU2810": "VCP → REC",
    "AZU2205": "CGH → BSB",
    "AZU4378": "REC → VCP",
    "AZU2987": "SSA → VCP",
    "AZU4554": "VCP → OPS",
    "AZU4338": "VCP → CGB",
    "AZU2912": "NAT → VCP",
    "AZU4867": "GYN → VCP",
    "AZU4696": "BSB → VCP",
    "AZU6001": "BSB → CGH",
    "AZU6004": "CGH → BSB",
    "AZU2474": "CNF → BPS",
    "AZU2603": "VCP → POA",
    "AZU2604": "BSB → CNF",
    "AZU2614": "VCP → GYN",
    "AZU2666": "SDU → BSB",
    "AZU2706": "VCP → BSB",
    "AZU2720": "LDB → CWB",
    "AZU2764": "GRU → REC",
    "AZU2798": "VCP → FOR",
    "AZU2801": "CNF → REC",
    "AZU2842": "GRU → POA",
    "AZU2901": "GIG → POA",
    "AZU2901": "GIG → POA",
    "AZU4002": "VIX → CNF",
    "AZU4005": "CNF → AJU",
    "AZU4010": "VCP → IOS",
    "AZU4030": "CGH → CNF",
    "AZU4039": "CNF → SDU",
    "AZU4076": "OPS → VCP",
    "AZU4079": "VCP → REC",
    "AZU4151": "POA → REC",
    "AZU4188": "CNF → VCP",
    "AZU4193": "GRU → SDU",
    "AZU4232": "GRU → REC",
    "AZU4268": "VCP → RAO",
    "AZU4360": "SLZ → REC",
    "AZU4469": "VCP → GIG",
    "AZU4651": "BEL → FOR",
    "AZU2904": "SSA → VCP",
    "GLO1234": "GRU → SDU",
    "GLO4056": "BSB → CGH",
    "GLO1587": "REC → GRU",
    "GLO1432": "CGH → GYN",
    "GLO1433": "GYN → CGH",
    "GLO1440": "GRU → OPS",
    "GLO1718": "BSB → SSA",
    "GLO7641": "AEP → GRU",    #AEP → BUENOS AIRES
    "GLO1555": "FOR → GRU",
    "GLO1437": "GYN → CGH",
    "GLO1635": "SSA → CGH",
    "GLO1845": "VIX → GIG",
    "GLO1863": "IGU → GIG",
    "GLO1880": "GIG → REC",
    "GLO1904": "SSA → GYN",
    "GLO2090": "GIG → POA",
    "GLO1002": "CGH → SDU",
    "GLO1003": "SDU → CGH",
    "GLO1004": "CGH → SDU",
    "GLO1005": "SDU → CGH",
    "GLO1104": "CGH → CWB",
    "GLO1183": "IGU → GRU",
    "GLO1209": "LDB → CGH",
    "GLO1213": "NVT → CGH",
    "GLO1270": "CGH → POA",
    "GLO1271": "POA → CGH",
    "GLO1272": "CGH → POA",
    "GLO1273": "POA → CGH",
    "GLO1304": "CGH → CNF",
    "GLO1305": "CNF → CGH",
    "GLO1306": "CGH → POA",
    "GLO1307": "POA → CGH",
    "GLO1338": "CGH → RAO",
    "GLO1642": "GRU → JPA",
    "GLO1692": "GRU → SSA",
    "GLO1653": "AJU → GRU",
    "GLO2050": "GIG → CGB",
    "GLO2178": "SSA → VIX",
    "TAM3421": "POA → GRU",
    "TAM3448": "GRU → MCZ",
    "TAM3456": "CGH → SSA",
    "TAM3891": "GRU → FOR",
    "TAM3637": "JPA → GRU",
    "TAM3555": "CNF → GRU",
    "TAM3737": "FLN → BSB",
    "TAM3596": "GRU → UBA",
    "TAM4667": "SJP → GRU",
    "TAM3118": "CGH → MCZ",
    "TAM3156": "CGH → POA",
    "TAM3157": "POA → CGH",
    "TAM3158": "CGH → POA",
    "TAM3159": "POA → CGH",
    "TAM3832": "GRU → PMW",
    "TAM3833": "PMW → GRU",
    "TAM3834": "GRU → PMW",
    "TAM3835": "PMW → GRU",
    "TAM3506": "GRU → FOR",
    "TAM3507": "FOR → GRU",
    "TAM3581": "CGH → BSB",
    "TAM3000": "CGH → BSB",
    "TAM3007": "BSB → CGH",
    "TAM3151": "POA → CGH",
    "TAM3173": "SDU → CGH",
    "TAM3203": "CGH → BSB",
    "TAM3228": "GRU → BEL",
    "TAM3229": "BEL → GRU",
    "TAM3237": "VCP → BSB",
    "TAM3254": "GRU → IOS",
    "TAM3353": "SSA → GRU",
    "TAM4596": "GRU → BEL",
    "TAM4602": "GRU → IOS",
    "TAM8194": "GRU → MIA",
    "TAP0082": "GRU → LIS"
}

#===============================================================================
# ORGÃO ATS / APROXIMAÇÃO
#===============================================================================
orgaos_ats = {
    "VCP": "Campinas Aproximação",
    "CGH": "São Paulo Aproximação",
    "REC": "Recife Aproximação",
    "SSA": "Salvador Aproximação",
    "GRU": "São Paulo, Guarulhos Aproximação",
    "SDU": "Rio de Janeiro Aproximação",
    "FOR": "Fortaleza Aproximação",
    "CNF": "Belo Horizonte Aproximação",
    "POA": "Porto Alegre Aproximação",
    "GIG": "Rio de Janeiro Aproximação",
    "BEL": "Belém Aproximação",
    "FLN": "Florianópolis Aproximação",
    "CWB": "Curitiba Aproximação",
    "IOS": "Ilhéus Aproximação",
    "GYN": "Goiânia Aproximação",
    "NAT": "Natal Aproximação",
    "RAO": "Ribeirão Preto Aproximação",
    "JPA": "João Pessoa Aproximação",
    "SJP": "São José do Rio Preto Aproximação",
    "SLZ": "São Luiz Aproximação",
    "LDB": "Londrina Aproximação",
    "PMW": "Palmas Aproximação",
    "UBA": "Uberlândia Aproximação",
    "CXJ": "Caxias Aproximação",
    "JOI": "Joinville Aproximação",
    "OPS": "Porto Seguro Aproximação",
    "AQA": "Araraquara Aproximação",
    "CGR": "Campo Grande Aproximação",
    "BPS": "Porto Seguro Aproximação",
    "AJU": "Aracaju Aproximação",
    "MCZ": "Maceió Aproximação",
    "MAO": "Manaus Aproximação",
    "VIX": "Vitoria Aproximação",
    "MIA": "Miami Aproximação",
    "LIS": "Lisboa Aproximação",
    "AEP": "Buenos Aires Aproximação"

}

#===============================================================================
# VOOS MONITORADOS PARA FONIA EM ÁUDIO
#===============================================================================
VOOS_MONITORADOS = {
    "AZU5093",
    "AZU2810",
    "AZU2205",
    "AZU4378",
    "AZU2987",
    "AZU4554",
    "AZU4338",
    "AZU2912",
    "AZU4867",
    "AZU4696",
    "AZU6001",
    "AZU6004",
    "AZU2474",
    "AZU2603",
    "AZU2604",
    "AZU2614",
    "AZU2666",
    "AZU2706",
    "AZU2720",
    "AZU2764",
    "AZU2798",
    "AZU2801",
    "AZU2842",
    "AZU2901",
    "AZU2901",
    "AZU4002",
    "AZU4005",
    "AZU4010",
    "AZU4030",
    "AZU4039",
    "AZU4076",
    "AZU4079",
    "AZU4151",
    "AZU4188",
    "AZU4193",
    "AZU4232",
    "AZU4268",
    "AZU4360",
    "AZU4469",
    "AZU4651",
    "AZU2904",
    "GLO1234",
    "GLO4056",
    "GLO1587",
    "GLO1432",
    "GLO1433",
    "GLO1440",
    "GLO1718",
    "GLO7641",
    "GLO1555",
    "GLO1437",
    "GLO1635",
    "GLO1845",
    "GLO1863",
    "GLO1904",
    "GLO2090",
    "GLO1002",
    "GLO1003",
    "GLO1004",
    "GLO1005",
    "GLO1104",
    "GLO1183",
    "GLO1209",
    "GLO1213",
    "GLO1270",
    "GLO1271",
    "GLO1272",
    "GLO1273",
    "GLO1304",
    "GLO1305",
    "GLO1306",
    "GLO1307",
    "GLO1338",
    "GLO1642",
    "GLO1653",
    "GLO1880",
    "GLO2050",
    "GLO2178",
    "TAM3421",
    "TAM3448",
    "TAM3456",
    "TAM3891",
    "TAM3637",
    "TAM3555",
    "TAM3737",
    "TAM3596",
    "TAM4667",
    "TAM3118",
    "TAM3156",
    "TAM3157",
    "TAM3158",
    "TAM3159",
    "TAM3832",
    "TAM3833",
    "TAM3834",
    "TAM3835",
    "TAM3506",
    "TAM3507",
    "TAM3581",
    "TAM3000",
    "TAM3007",
    "TAM3151",
    "TAM3173",
    "TAM3203",
    "TAM3228",
    "TAM3229",
    "TAM3237",
    "TAM3254",
    "TAM3353",
    "TAM4596",
    "TAM4602",
    "TAM8194"
}

#===============================================================================
# COORDENADAS DAS ROTAS PARA DESENHO NO MAPA
#===============================================================================
coordenadas_rotas = {

    "VIX": [-20.2580, -40.2860],
    "VCP": [-23.0074, -47.1345],
    "REC": [-8.1264, -34.9236],
    "CGH": [-23.6261, -46.6566],
    "BSB": [-15.8711, -47.9186],
    "SSA": [-12.9086, -38.3225],
    "GRU": [-23.4356, -46.4731],
    "SDU": [-22.9104, -43.1631],
    "FOR": [-3.7763, -38.5326],
    "CNF": [-19.6244, -43.9719],
    "POA": [-29.9944, -51.1714],
    "GIG": [-22.8090, -43.2506],
    "BEL": [-1.3792, -48.4763],
    "FLN": [-27.6705, -48.5527],
    "CWB": [-25.5285, -49.1758],
    "IGU": [-25.6003, -54.4850],
    "GYN": [-16.6320, -49.2207],
    "NAT": [-5.9114, -35.2477],
    "RAO": [-21.1342, -47.7741],
    "JPA": [-7.1458, -34.9486],
    "SJP": [-20.8166, -49.4065],
    "SLZ": [-2.5854, -44.2341],
    "LDB": [-23.3336, -51.1301],
    "PMW": [-10.2417, -48.3528],  #falta aeroporto
    "UBA": [-21.1271, -42.9428],  #falta aeroporto
    "IOS": [-14.8150, -39.0332],  #falta aeroporto
    "PET": [-31.7184, -52.3277],
    "CXJ": [-29.1971, -51.1875],
    "JOI": [-26.2245, -48.7974],
    "NVT": [-26.8797, -48.6514],
    "OPS": [-11.8850, -55.5861],
    "AQA": [-21.8044, -48.1403],
    "CGR": [-20.4687, -54.6725],
    "BPS": [-16.4386, -39.0808],
    "AJU": [-10.9840, -37.0703],
    "MCZ": [-9.5108, -35.7916],
    "MAO": [-3.0408, -60.0506],

    # INTERNACIONAIS
    "AEP": [-34.5592, -58.4156],  # Buenos Aires
    "MIA": [25.7959, -80.2870],   # Miami
    "LIS": [38.7742, -9.1342]     # Lisboa
}

#===============================================================================
# LISTA DE ÁUDIOS GERADOS
#===============================================================================
audios_gerados = []
# ==============================================================================
# INSERÇÃO DAS AERONAVES NO MAPA
# ==============================================================================

for _, row in df_brasil.iterrows():

    callsign = str(row['callsign']).strip().upper()

    print(f"CALLSIGN DETECTADO: [{callsign}]")

    velocidade = row['velocity_kmh']
    altitude = row['altitude_ft']
    pais = row['origin_country']

    rota = rotas_conhecidas.get(callsign, "Rota não identificada")

    fonia = gerar_fonia(
        callsign,
        altitude,
        velocidade,
        rota
    )

    if rota != "Rota não identificada":
        desenhar_rota(
          mapa,
          rota,
          row['latitude'],
          row['longitude']
        )

    popup_texto = f"""
    <b>CALLSIGN:</b><br>
    {callsign}<br>

    <b>PÁIS:</b><br>
    {pais}<br>

    <b>ROTA:</b><br>
    {rota}<br>

    <b>VELOCIDADE:</b><br>
    {velocidade} km/h<br>

    <b>ALTITUDE:</b><br>
    {altitude:,.0f} pés<br>

    <hr>

    <b>🎧 FONIA:</b><br>
    {fonia}
    """

    angulo = (row['true_track'] -45) if pd.notna(row['true_track']) else 0

    icone_aviao = folium.DivIcon(
        html=f"""
        <div style="
              transform: rotate({angulo}deg);
              font-size: 24px;
              color: red;
              margin-left: -8px;
              margin-top: -8px;
        ">
           ✈
        </div>
        """
    )

    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=popup_texto,
        tooltip=callsign,
        icon=icone_aviao
    ).add_to(mapa)

#===============================================================================
# DISPARO DE ÁUDIO DA FONIA
#===============================================================================
    if callsign.strip() in VOOS_MONITORADOS:
        print(f"\n🎧 FONIA ATIVA:{callsign}")
        audios_gerados.append(
        (callsign,fonia)
    )

# ==============================================================================
# INSERÇÃO DOS AEROPORTOS NO MAPA
# ==============================================================================

print(f"\n{YELLOW}🛫 Inserindo aeroportos no mapa...{RESET}")

# ==============================================================================
# BASE DE AEROPORTOS BRASILEIROS
# ==============================================================================

aeroportos = [

    # ==========================================================================
    # SUDESTE
    # ==========================================================================
    {
        "icao": "SBGR",
        "iata": "GRU",
        "nome": "Aeroporto Internacional de Guarulhos",
        "cidade": "Guarulhos",
        "estado": "São Paulo",
        "lat": -23.4356,
        "long": -46.4731
    },

    {
        "icao": "SBSP",
        "iata": "CGH",
        "nome": "Aeroporto de Congonhas",
        "cidade": "São Paulo",
        "estado": "São Paulo",
        "lat": -23.6261,
        "long": -46.6566
    },

    {
        "icao": "SBKP",
        "iata": "VCP",
        "nome": "Aeroporto Internacional de Campinas",
        "cidade": "Campinas",
        "estado" : "São Paulo",
        "lat": -23.0074,
        "long": -47.1345
    },

    {
        "icao": "SBAQ",
        "iata": "AQA",
        "nome": "Aeroporto Estadual Bartholomeu de Gusmão",
        "cidade": "Araraquara",
        "estado": "São Paulo",
        "lat": -21.8044,
        "long": -48.1403
    },

    {
        "icao": "SBRP",
        "iata": "RAO",
        "nome": "Aeroporto Estadual Dr. Leite Lopes",
        "cidade": "Ribeirão Preto",
        "estado": "São Paulo",
        "lat": -21.1342,
        "long": -47.7741

    },

    {
        "icao": "SBSR",
        "iata": "SJR",
        "nome": "Aeroporto Prof. Eriberto Manoel Reino",
        "cidade": "São José do Rio Preto",
        "estado": "São Paulo",
        "lat": -20.8166,
        "long": -49.4065

    },

    {
        "icao": "SBGL",
        "iata": "GIG",
        "nome": "Aeroporto do Galeão",
        "cidade": "Rio de Janeiro",
        "estado": "Rio de Janeiro",
        "lat": -22.8090,
        "long": -43.2506

    },

    {
        "icao": "SBRJ",
        "iata": "SDU",
        "nome": "Aeroporto Santos Dumont",
        "cidade": "Rio de Janeiro",
        "estado": "Rio de Janeiro",
        "lat": -22.9104,
        "long": -43.1631
    },

    {
        "icao": "SBVT",
        "iata": "VIX",
        "nome": "Aeroporto Eurico de Aguiar Salles",
        "cidade": "Vitoria",
        "estado": "Espirito Santo",
        "lat": -20.2580,
        "long": -40.2860

    },

    {
        "icao": "SBCF",
        "iata": "CNF",
        "nome": "Aeroporto de Confins",
        "cidade": "Belo Horizonte",
        "estado": "Minas Gerais",
        "lat": -19.6244,
        "long": -43.9719
    },

    # ==========================================================================
    # CENTRO-OESTE
    # ==========================================================================
    {
        "icao": "SBBR",
        "iata": "BSB",
        "nome": "Aeroporto Internacional de Brasília",
        "cidade": "Brasília",
        "estado": "Distrito Federal",
        "lat": -15.8711,
        "long": -47.9186

    },

    {
        "icao": "SBGO",
        "iata": "GYN",
        "nome": "Aeroporto Santa Genoveva",
        "cidade": "Goiânia",
        "estado": "Goiás",
        "lat": -16.6320,
        "long": -49.2207
    },

    {
        "icao": "SBCG",
        "iata": "CGR",
        "nome": "Aeroporto de Campo Grande",
        "cidade": "Campo Grande",
        "estado": "Mato Grosso",
        "lat": -20.4687,
        "long": -54.6725
    },

    {
        "icao": "SBSI",
        "iata": "OPS",
        "nome": "Aeroporto Presidente João Batista Figueiredo",
        "cidade": "Sinop",
        "estado": "Mato Grosso",
        "lat": -11.8850,
        "long": -55.5861
    },
    # ==========================================================================
    # NORDESTE
    # ==========================================================================
    {
        "icao": "SBPS",
        "iata": "BPS",
        "nome": "Aeroporto Porto Seguro",
        "cidade": "Porto Seguro",
        "estado": "Bahia",
        "lat": -16.4386,
        "long": -39.0808
    },

    {
        "icao": "SBSV",
        "iata": "SSA",
        "nome": "Aeroporto Internacional Dep. Luiz Eduardo Magalhães",
        "cidade": "Salvador",
        "estado": "Bahia",
        "lat": -12.9086,
        "long": -38.3225
    },

    {
        "icao": "SBAR",
        "iata": "AJU",
        "nome": "Aeroporto Santa Maria",
        "cidade": "Aracaju",
        "estado": "Sergipe",
        "lat": -10.9840,
        "long": -37.0703
    },

    {
        "icao": "SBMO",
        "iata": "MCZ",
        "nome": "Aeroporto Zumbi dos Palmares",
        "cidade": "Maceió",
        "estado": "Alagoas",
        "lat": -9.5108,
        "long": -35.7916

    },

    {
        "icao": "SBJP",
        "iata": "JPA",
        "nome": "Aeroporto Presidente Castro Pinto",
        "cidade": "João Pessoa",
        "estado": "Paraíba",
        "lat": -7.1458,
        "long": -34.9486
    },

    {
        "icao": "SBNT",
        "iata": "NAT",
        "nome": "Aeroporto Augusto Severo",
        "cidade": "Natal",
        "estado": "Rio Grande do Norte",
        "lat": -5.9114,
        "long": -35.2477
    },

    {
        "icao": "SBFZ",
        "iata": "FOR",
        "nome": "Aeroporto Internacional Pinto Martins",
        "cidade": "Fortaleza",
        "estado": "Ceará",
        "lat": -3.7763,
        "long": -38.5326
    },

    {
        "icao": "SBRF",
        "iata": "REC",
        "nome": "Aeroporto Internacional de Recife",
        "cidade": "Recife",
        "estado": "Pernambuco",
        "lat": -8.1264,
        "long": -34.9236
    },

    {
        "icao": "SBSL",
        "iata": "SLZ",
        "nome": "Aeroporto Internacional Marechal Cunha Machado",
        "cidade": "São Luiz",
        "estado": "Maranhão",
        "lat": -2.5854,
        "long": -44.2341

    },

    # ==========================================================================
    # NORTE
    # ==========================================================================
    {
        "icao": "SBBE",
        "iata": "BEL",
        "nome": "Aeroporto Internacional Val de Cans",
        "cidade": "Belém",
        "estado": "Pará",
        "lat": -1.3792,
        "long": -48.4763

    },

    {
        "icao": "SBEG",
        "iata": "MAO",
        "nome": "Aeroporto Internacional Eduardo Gomes",
        "cidade": "Manaus",
        "estado": "Amazonas",
        "lat": -3.0408,
        "long": -60.0506
    },



    # ==========================================================================
    # SUL - PARANÁ
    # ==========================================================================
    {
        "icao": "SBCT",
        "iata": "CWB",
        "nome": "Aeroporto Afonso Pena",
        "cidade": "Curitiba",
        "estado": "Paraná",
        "lat": -25.5285,
        "long": -49.1758

    },

    {
        "icao": "SBFI",
        "iata": "IGU",
        "nome": "Aeroporto de Foz do Iaguaçu",
        "cidade": "Foz do Iguaçu",
        "estado": "Paraná",
        "lat": -25.6003,
        "long": -54.4850

    },

    {
        "icao": "SBLO",
        "iata": "LDB",
        "nome": "Aeroporto de Londrina",
        "cidade": "Londrina",
        "estado": "Paraná",
        "lat": -23.3336,
        "long": -51.1301
    },

    # ==========================================================================
    # SUL - SANTA CATARINA
    # ==========================================================================
    {
        "icao": "SBFL",
        "iata": "FLN",
        "nome": "Aeroporto de Florianópolis",
        "cidade": "Florianópolis",
        "estado": "Santa Catarina",
        "lat": -27.6705,
        "long": -48.5527
    },

    {
        "icao": "SBNF",
        "iata": "NVT",
        "nome": "Aeroporto de Navegantes",
        "cidade": "Navegantes",
        "estado": "Santa Catarina",
        "lat": -26.8797,
        "long": -48.6514
    },

    {
        "icao": "SBJV",
        "iata": "JOI",
        "nome": "Aeroporto de Joinville",
        "cidade": "Joinville",
        "estado": "Santa Catarina",
        "lat": -26.2245,
        "long": -48.7974
    },
    # ==========================================================================
    # SUL - RIO GRANDE DO SUL
    # ==========================================================================
    {
        "icao": "SBPA",
        "iata": "POA",
        "nome": "Aeroporto Salgado Filho",
        "cidade": "Porto Alegre",
        "estado": "Rio Grande do Sul",
        "lat": -29.9944,
        "long": -51.1714
    },

    {
        "icao": "SBCX",
        "iata": "CXJ",
        "nome": "Aeroporto de Caxias do Sul",
        "cidade": "Caxias do Sul",
        "estado": "Rio Grande do Sul",
        "lat": -29.1971,
        "long": -51.1875
    },

    {
        "icao": "SBPK",
        "iata": "PET",
        "nome": "Aeroporto de Pelotas",
        "cidade": "Pelotas",
        "estado": "Rio Grande do Sul",
        "lat": -31.7184,
        "long": -52.3277
    },
#===============================================================================
# INTERNACIONAIS
#===============================================================================
    {
        "icao": "SABE",
        "iata": "AEP",
        "nome": "Aeroparque Jorge Newbery",
        "cidade": "Buenos Aires",
        "estado": "Argentina",
        "lat": -34.5592,
        "long": -58.4156
    },

    {
        "icao": "KMIA",
        "iata": "MIA",
        "nome": "Miami International Airport",
        "cidade": "Miami",
        "estado": "Estados Unidos",
        "lat": 25.7959,
        "long": -80.2870
    },

    {
        "icao": "LPPT",
        "iata": "LIS",
        "nome": "Aeroporto Humberto Delgado",
        "cidade": "Lisboa",
        "estado": "Portugal",
        "lat": 38.7742,
        "long": -9.1342
    },

]


# ==============================================================================
# INSERÇÃO DOS AEROPORTOS NO MAPA E CONSULTA METEOROLÓGICA
# ==============================================================================

print(f"\n{YELLOW}🛫 Inserindo aeroportos no mapa e consultando clima...{RESET}")

for aeroporto in aeroportos:
    time.sleep(1)

    clima = obter_clima(
        aeroporto['lat'],
        aeroporto['long']
    )

    condicao = traduzir_weathercode(
        clima['codigo']
    )

    # POPUP METEOROLÓGICO for each airport
    popup_aeroporto = f"""
    <b>AEROPORTO:</b> {aeroporto['nome']}<br>
    <b>CIDADE:</b> {aeroporto['cidade']}<br>
    <b>ESTADO:</b> {aeroporto['estado']}<br>
    <b>ICAO:</b> {aeroporto['icao']}<br>
    <b>IATA:</b> {aeroporto['iata']}<br>
    <hr>
    <b>🌡TEMPERATURA:</b> {clima['temperatura']} °C<br>
    <b>💨VENTO:</b> {clima['vento']} km/h<br>
    <b>☁CONDIÇÃO:</b> {condicao}
    """

    folium.Marker(
        location=[aeroporto['lat'], aeroporto['long']],
        popup=popup_aeroporto,
        tooltip=f"{aeroporto['icao']} / {aeroporto['iata']}",
        icon=folium.Icon(
            color='green',
            icon='info-sign'
            )
    ).add_to(mapa)

print(f"{GREEN}✅ Aeroportos adicionados com sucesso!{RESET}")

# ==============================================================================
# EXIBIÇÃO DO MAPA
# ==============================================================================

print(f"{GREEN}✅ Mapa operacional gerado com sucesso!{RESET}")
display(mapa)

# =========================================================
# REPRODUÇÃO DOS ÁUDIOS DE FONIA
# =========================================================

for callsign_audio, texto_audio in audios_gerados:

    print(f"\n🎧 REPRODUZINDO FONIA: {callsign_audio}")

    texto_limpo = (
        texto_audio
        .replace("<br>", " ")
        .replace("🎧", "")
    )

    audio = falar_fonia(

        texto_limpo,

        arquivo=f"{callsign_audio}.mp3"
    )

    print(f"{callsign_audio}.mp3 criado")

    display(audio)
# ==============================================================================
# ENCERRAMENTO
# ==============================================================================

print(f"\n{CYAN}{'='*90}{RESET}")
print(f"{BOLD} FLIGHTOPS ANALYTCS BR - MONITORAMENTO FINALIZADO{RESET}")
print(f"{CYAN}{'='*90}{RESET}")